## 📚 Import Required Libraries

This cell imports the Python libraries used throughout the notebook:

- **Pandas** helps load and organize the car dataset.
- **Matplotlib** and **Seaborn** are available for plotting.

These imports prepare the environment for data loading and visualization.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px



## 📂 Load the Cleaned Dataset

This cell reads the cleaned used-cars dataset into a Pandas DataFrame named `df`.

- The CSV file must be available in the notebook's working directory.
- `df.head()` displays the first five rows so we can quickly inspect the dataset and confirm that it loaded correctly.

In [2]:

import pandas as pd

df = pd.read_csv("cleaned_used_cars_dataset.csv")

df.head()


,listing_id,brand,model,year,mileage_km,transmission,fuel_type,engine_cc,body_type,color,city,seller_type,accident_history,previous_owners,condition_rating,photos_count,price_egp,sold_within_30_days
0,CAR-52762,MG,Zs,2021,64900.0,Automatic,Diesel,1500.0,SUV,Black,Giza,Individual,No,1.0,Fair,8.0,655000,Yes
1,CAR-52678,Kia,Cerato,2023,42100.0,Automatic,Diesel,1600.0,Sedan,Grey,Giza,Individual,No,1.0,Good,12.0,1012000,No
2,CAR-52822,Skoda,Octavia,2009,224500.0,Automatic,Petrol,1400.0,Sedan,Silver,Banha,Dealer,Yes,2.0,Fair,11.0,229000,No
3,CAR-51133,Fiat,Tipo,2019,103800.0,Automatic,Petrol,1600.0,Sedan,Silver,Alexandria,Individual,No,2.0,Fair,7.0,319000,No
4,CAR-51465,Mitsubishi,Lancer,2022,50200.0,Manual,Hybrid,1600.0,Sedan,White,Cairo,Individual,No,2.0,Fair,8.0,425000,No


## 💰 Top 15 Brands by Median Price

This horizontal bar chart compares the **15 brands with the highest median car prices**.

- The median price is used instead of the average because it is less affected by unusually expensive cars.
- The bars are ordered by median price, making the brands easier to compare.
- Each bar displays its median price in EGP, and hovering over it provides more detail.

In [3]:

import plotly.express as px

brand_prices = df.groupby("brand")["price_egp"].median()
brand_prices = brand_prices.sort_values().tail(15)

fig = px.bar(
    x=brand_prices.values,
    y=brand_prices.index,
    orientation="h",
    title="Top 15 Brands by Median Price",
    labels={
        "x": "Median Price (EGP)",
        "y": "Brand"
    },
    text=brand_prices.values
)

fig.update_traces(
    texttemplate="%{text:,.0f} EGP",
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Median Price: %{x:,.2f} EGP<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Median Price (EGP)",
    yaxis_title="Brand",
    yaxis=dict(categoryorder="total ascending"),
    height=600,
    showlegend=False
)

fig.show()

## ⛽ Fuel Type by Transmission

This stacked bar chart shows how transmission types are distributed across fuel types.

- `pd.crosstab()` counts the records for each combination of fuel type and transmission.
- The counts are reshaped into a long format so Plotly can color each transmission category separately.
- Comparing the stacked bars helps reveal which fuel/transmission combinations occur most often in the dataset.

In [4]:

import pandas as pd
import plotly.express as px

fuel = pd.crosstab(df["fuel_type"], df["transmission"])

fuel_long = fuel.reset_index().melt(
    id_vars="fuel_type",
    var_name="transmission",
    value_name="count"
)

fig = px.bar(
    fuel_long,
    x="fuel_type",
    y="count",
    color="transmission",
    title="Fuel Type by Transmission",
    labels={
        "fuel_type": "Fuel Type",
        "count": "Count",
        "transmission": "Transmission"
    },
    barmode="stack",
    text="count"
)

fig.update_traces(
    hovertemplate=(
        "Fuel Type: %{x}<br>"
        "Transmission: %{fullData.name}<br>"
        "Count: %{y}<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    height=500,
    legend_title_text="Transmission",
    hovermode="x unified"
)

fig.show()

## 🚗 Cars Sold Within 30 Days

This pie chart displays the share of cars in each **`sold_within_30_days`** category.

- The chart uses the category counts to calculate each slice's percentage.
- Labels and percentages are shown inside the slices.
- Hover over a slice to see its status, count, and percentage.

The chart describes the dataset's recorded sales-status categories; it does not by itself explain why a car sold quickly or slowly.

In [5]:

import plotly.express as px

sales_counts = df["sold_within_30_days"].value_counts().dropna()

fig = px.pie(
    names=sales_counts.index,
    values=sales_counts.values,
    title="Cars Sold Within 30 Days",
    hole=0,  # Pie chart عادي، خليها 0.4 لو عايز Donut
)

fig.update_traces(
    textinfo="label+percent",
    textposition="inside",
    hovertemplate=(
        "Status: %{label}<br>"
        "Count: %{value}<br>"
        "Percentage: %{percent}<extra></extra>"
    ),
    sort=False
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()


## 📉 Price vs Mileage

This scatter plot explores the relationship between a car's mileage and its price.

- Each point represents a car.
- The horizontal axis shows mileage in kilometers, while the vertical axis shows price in EGP.
- Color represents whether the car was sold within 30 days.
- Hovering over a point reveals additional details, including brand, fuel type, and transmission.

Look for overall patterns and clusters, but remember that mileage is only one factor that may influence price.

In [6]:

import plotly.express as px

fig = px.scatter(
    df,
    x="mileage_km",
    y="price_egp",
    color="sold_within_30_days",
    title="Price vs Mileage",
    labels={
        "mileage_km": "Mileage (km)",
        "price_egp": "Price (EGP)",
        "sold_within_30_days": "Sold Within 30 Days"
    },
    hover_data=["brand", "fuel_type", "transmission"]
)

fig.update_layout(
    template="plotly_white",
    height=550,
    xaxis_title="Mileage (km)",
    yaxis_title="Price (EGP)",
    legend_title_text="Sold Within 30 Days"
)

fig.update_traces(
    marker=dict(size=8, opacity=0.7),
    hovertemplate=(
        "Mileage: %{x:,.0f} km<br>"
        "Price: %{y:,.0f} EGP<br>"
        "Sold Within 30 Days: %{fullData.name}"
        "<extra></extra>"
    )
)

fig.show()


## 🌐 3D Analysis: Car Year, Mileage & Price

This interactive 3D scatter plot examines three numeric dimensions together:

- **X-axis:** manufacturing year.
- **Y-axis:** mileage in kilometers.
- **Z-axis:** price in EGP.
- **Color:** whether the car was sold within 30 days.

Hover over a point to inspect the car's brand, model, year, mileage, price, city, and sales status. Rotating the chart can help reveal patterns that are harder to see in a 2D plot.

Use the visualization to explore possible relationships; it does not establish that any one variable causes another.

In [9]:

import plotly.express as px

fig = px.scatter_3d(
    df,
    x="year",
    y="mileage_km",
    z="price_egp",
    color="sold_within_30_days",
    hover_data=["brand", "model", "city"],
    title="3D Analysis: Car Year, Mileage & Price",
    labels={
        "year": "Manufacturing Year",
        "mileage_km": "Mileage (km)",
        "price_egp": "Price (EGP)",
        "sold_within_30_days": "Sold Within 30 Days"
    },
    opacity=0.8
)

fig.update_traces(
    marker=dict(size=4),
    hovertemplate=(
        "<b>%{customdata[0]} %{customdata[1]}</b><br>"
        "Year: %{x}<br>"
        "Mileage: %{y:,.0f} km<br>"
        "Price: %{z:,.0f} EGP<br>"
        "City: %{customdata[2]}<br>"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=700,
    scene=dict(
        xaxis_title="Manufacturing Year",
        yaxis_title="Mileage (km)",
        zaxis_title="Price (EGP)"
    ),
    legend_title_text="Sold Within 30 Days"
)

fig.show()